# fPCA-SYNTH / RPCA-SYNTH — West Germany application

This notebook implements the fPCA-SYNTH comparison for the West Germany application using the original, unscaled GDP outcomes.

The implementation is based on the `mlsynth` package:
https://github.com/jgreathouse9/mlsynth

**Before running:** install the Python dependencies listed in `requirements_python.txt` and make sure that `repgermany.dta` is available in the `data/` directory.

## Imports

In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

from pathlib import Path
import importlib.metadata

import numpy as np
import pandas as pd

from mlsynth import CLUSTERSC
from mlsynth.config_models import CLUSTERSCConfig

## 1. User settings

In [2]:
INPUT_DATA_PATH = Path("data/repgermany.dta")
OUTPUT_DIR = Path("results/realdata/fPCA_SYNTH")

PRE_START = 1960
PRE_END = 1989
POST_START = 1990
POST_END = 2003

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data preparation

In [3]:
data = pd.read_stata(INPUT_DATA_PATH)

data["country"] = data["country"].astype(str)
data["year"] = data["year"].astype(int)
data["gdp"] = pd.to_numeric(data["gdp"])

data = data.loc[
    data["year"].between(PRE_START, POST_END),
    ["country", "year", "gdp"],
].copy()

if data.duplicated(["country", "year"]).any():
    raise ValueError("Duplicate country-year observations found.")

if data[["country", "year", "gdp"]].isna().any().any():
    raise ValueError("Missing values found in the analysis panel.")

# Long-format panel expected by mlsynth.
data["treat"] = (
    (data["country"] == "West Germany")
    & (data["year"] >= POST_START)
).astype(int)

## 3. Fit fPCA-SYNTH

In [4]:
config = CLUSTERSCConfig(
    df=data,
    outcome="gdp",
    treat="treat",
    unitid="country",
    time="year",
    method="RPCA",
    cluster=True,
    Frequentist=True,
    ROB="PCP",
    display_graphs=False,
    save=False,
)

model = CLUSTERSC(config)
results = model.fit()

rpca_res = results.sub_method_results["RPCA"]

## 4. Donor weights

In [5]:
weight_dict = rpca_res.weights.donor_weights
selected_weights = pd.Series(
    weight_dict,
    name="weight",
    dtype=float,
)

all_donors = sorted(
    set(data["country"]) - {"West Germany"}
)

weight_table = pd.DataFrame(
    {"country": all_donors}
)

weight_table["selected"] = weight_table["country"].isin(
    selected_weights.index
)

weight_table["weight"] = weight_table["country"].map(
    selected_weights
)

print("\n=== fPCA-SYNTH selected donor weights ===")
print(
    weight_table.loc[weight_table["selected"]]
    .sort_values("weight", ascending=False)
    .to_string(index=False)
)


=== fPCA-SYNTH selected donor weights ===
    country  selected  weight
     Norway      True   0.485
     France      True   0.354
New Zealand      True   0.296
    Austria      True   0.023


## 5. Counterfactual path, Pre-RMSE, and ATT

In [6]:
ts = rpca_res.time_series

years = np.asarray(ts.time_periods).ravel().astype(int)
observed = np.asarray(ts.observed_outcome).squeeze()
counterfactual = np.asarray(
    ts.counterfactual_outcome
).squeeze()

if not (
    len(years)
    == len(observed)
    == len(counterfactual)
):
    raise ValueError(
        "Unexpected mismatch in fPCA-SYNTH trajectory lengths."
    )

trajectory = pd.DataFrame(
    {
        "year": years,
        "observed": observed,
        "counterfactual": counterfactual,
    }
)

trajectory["effect"] = (
    trajectory["observed"]
    - trajectory["counterfactual"]
)

pre_mask = trajectory["year"].between(
    PRE_START,
    PRE_END,
)

post_mask = trajectory["year"].between(
    POST_START,
    POST_END,
)

pre_rmse = np.sqrt(
    np.mean(
        trajectory.loc[
            pre_mask,
            "effect",
        ].to_numpy() ** 2
    )
)

att = trajectory.loc[
    post_mask,
    "effect",
].mean()

summary = pd.DataFrame(
    {
        "method": ["fPCA-SYNTH"],
        "pre_RMSE": [pre_rmse],
        "post_ATT": [att],
    }
)

print("\n=== fPCA-SYNTH summary ===")
print(summary.to_string(index=False))


=== fPCA-SYNTH summary ===
    method  pre_RMSE     post_ATT
fPCA-SYNTH  88.60303 -1500.933286


## 6. Export

In [7]:
weight_table.to_csv(
    OUTPUT_DIR / "fPCA_SYNTH_weights.csv",
    index=False,
)

trajectory.to_csv(
    OUTPUT_DIR / "fPCA_SYNTH_trajectory.csv",
    index=False,
)

summary.to_csv(
    OUTPUT_DIR / "fPCA_SYNTH_summary.csv",
    index=False,
)

with open(
    OUTPUT_DIR / "python_environment.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write(
        "mlsynth source commit used in the original analysis: "
        "ea53dde7c95fd165f5736c5ce063dd8c4cb8bf30\n"
    )
    try:
        f.write(
            "mlsynth installed version: "
            + importlib.metadata.version("mlsynth")
            + "\n"
        )
    except importlib.metadata.PackageNotFoundError:
        pass
    f.write("numpy: " + np.__version__ + "\n")
    f.write("pandas: " + pd.__version__ + "\n")